In [ ]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
from datasets import Dataset, DatasetDict, concatenate_datasets
import torch
from torch.utils.data import DataLoader
import glob
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (set_seed,
                          TrainingArguments,
                          Trainer,
                          Qwen3ForCausalLM,
                          AutoTokenizer,
                          DataCollatorWithPadding)
from tqdm import tqdm
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

In [ ]:
CURRENT_SEED = 42

set_seed(CURRENT_SEED)

In [ ]:
# Extract tabfact zip if notebook is in google colab

import zipfile

zip_path = '/content/tabfact_data.zip'
extract_path = '/content/'

print("Unzipping dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping completed! Files extracted to:", extract_path)

In [ ]:
# Set data path, if notebook is in google colab use content, if not it is in the directory above

# # Local repository path
# DATA_PATH = "../tabfact_data"

# Colab path
DATA_PATH = "/content/tabfact_data"

In [ ]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [ ]:
# Load in table dataframe

# DOESN'T WORK IN COLAB BUT FASTER IN LOCAL REPO
df_tables = (
    pl.read_csv(
        "../tabfact_data/tables/*.csv",
        has_header=False,
        separator="\1", # Non-existent delimiter so full line/file stays intact
        new_columns=["table_content"],
        include_file_paths="file_name",
    )
    .group_by("file_name", maintain_order=True) # Group all the table lines together 
    .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
    .to_pandas()
)

# csv_files = sorted(glob.glob(f"{DATA_PATH}/tables/*.csv"))

# df_tables = (
#     pl.concat([
#         pl.read_csv(
#             file,
#             has_header=False,
#             separator="\1",
#             new_columns=["table_content"],
#         ).with_columns(pl.lit(Path(file).name).alias("file_name"))
#         for file in csv_files
#     ])
#     .group_by("file_name", maintain_order=True)  # Group all the table lines together
#     .agg(pl.col("table_content").str.join("\n"))  # Concatenate table lines together
#     .to_pandas()
# )

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

In [ ]:
# Defining column labelled rows
print(df_tables.shape)

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        ", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

In [ ]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe(f"{DATA_PATH}/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1 = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r1_training_all.json")
df_r2 = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r2_training_all.json")


df_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/train_examples.json")
df_test = load_and_convert_data_to_dataframe(f"{DATA_PATH}/test_examples.json")
df_validation = load_and_convert_data_to_dataframe(f"{DATA_PATH}/val_examples.json")


In [ ]:
NO_THINKING_PROMPT = "You are a binary classifier. Respond only with ENTAILS or REFUTES. If the CLAIM is entailed by the TABLE, respond with ENTAILS. If the CLAIM is refuted by the TABLE respond with REFUTES."
THINKING_PROMPT = "You are a binary classifier. If the CLAIM is entailed by the TABLE, respond with ENTAILS. If the CLAIM is refuted by the TABLE respond with REFUTES. Reason carefully inside <think></think> before deciding. After </think>, output ONLY ENTAILS or REFUTES with no extra text. "

In [ ]:
def get_table_claim_df(df, table_df_col, table_before_claim=True, table_prefix="TABLE: ", claim_prefix="CLAIM: ", table_claim_separator=" | ", before_evidence_text=""):
    df_table_claim = pd.merge(
        df_tables[["file_name", table_df_col]],
        df[["table_id", "statement", "label"]],
        left_on="file_name",
        right_on="table_id",
        how="inner"
    ).reset_index(drop=True)
    if table_before_claim:
      before_evidence_text += "Here is the TABLE and then the CLAIM: \n" if before_evidence_text != "" else ""
      df_table_claim["table_claim"] = (
        before_evidence_text +
        table_prefix +
        df_table_claim[table_df_col] +
        table_claim_separator +
        claim_prefix +
        df_table_claim["statement"]
      )
    else:
        before_evidence_text += "Here is the CLAIM and then the TABLE: \n" if before_evidence_text != "" else ""
        df_table_claim["table_claim"] = (
          before_evidence_text +
          claim_prefix +
          df_table_claim["statement"] +
          table_claim_separator +
          table_prefix +
          df_table_claim[table_df_col]
        )

    return df_table_claim[["table_claim", "label"]].copy()

In [ ]:
df_train_table_claim = get_table_claim_df(df_training, "table_content", table_before_claim=False, table_claim_separator="\n")
df_val_table_claim = get_table_claim_df(df_validation, "table_content", table_before_claim=False, table_claim_separator="\n")
df_test_table_claim = get_table_claim_df(df_test, "table_content", table_before_claim=False, table_claim_separator="\n")
df_r1_table_claim = get_table_claim_df(df_r1, "table_content", table_before_claim=False, table_claim_separator="\n")
df_r2_table_claim = get_table_claim_df(df_r2, "table_content", table_before_claim=False, table_claim_separator="\n")

print(df_val_table_claim["table_claim"].iloc[0])

print(df_val_table_claim.shape)

In [3]:
PROMPT = "You are a binary classifier. Respond only with ENTAILS or REFUTES. If the CLAIM is entailed by the TABLE, respond with ENTAILS. If the CLAIM is refuted by the TABLE respond with REFUTES."

In [ ]:
example_messages = []

# Preprocess function that correctly applies the chat template to each entry and returns the tokenized batch
def preprocess_batch(entries, with_system_prompt):
    formatted_data = []
    for table_claim, label in zip(entries["table_claim"], entries["label"]):
        prompt = []

        text_label = "ENTAILS" if label == 1 else "REFUTES"

        if with_system_prompt:
            prompt.append({
                "role": "system",
                "content": PROMPT
            })

        prompt.append({
            "role": "user",
            "content": table_claim
        })


        completion = [{
            "role": "assistant",
            "content": text_label
        }]

        data_formatted = {
            "prompt": prompt,
            "completion": completion
        }

        formatted_data.append(data_formatted)

    return formatted_data


train_dataset = Dataset.from_pandas(df_train_table_claim)
val_dataset = Dataset.from_pandas(df_val_table_claim)
test_dataset = Dataset.from_pandas(df_test_table_claim)

datasets = DatasetDict({
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset
})

preprocessed_datasets = datasets.map(lambda x: preprocess_batch(x, False), batched=True, remove_columns="table_claim")

In [ ]:
print(next(iter(preprocessed_datasets["train"])))

In [ ]:
# Lora and SFT configurations
lora_config = LoraConfig(
    r=16,
    lora_alpha=32, # should be 2r
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

sft_config = SFTConfig(
    output_dir="./qwen3-0.6b-tabfact-verif",
    model_init_kwargs={"dtype": torch.bfloat16},
    max_length=2048,        
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # Effective batch size = 16
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    chat_template_kwargs={"enable_thinking": False},  
    logging_steps=10,
)

In [ ]:
trainer = SFTTrainer(
    Model="Qwen/Qwen3-0.6B",
    train_dataset=preprocessed_datasets["train"],
    peft_config=lora_config
    sft_config=sft_config
)

trainer.train()

In [ ]:
def get_reduced_random_test_dataset(df, size):
  entailed_test_dataset = Dataset.from_pandas(df[df['label'] == 1])
  refuted_test_dataset = Dataset.from_pandas(df[df['label'] == 0])

  generator = np.random.default_rng(CURRENT_SEED)
  entailed_random_indices = generator.choice(len(entailed_test_dataset), size=int(size/2), replace=False)
  refuted_random_indices = generator.choice(len(refuted_test_dataset), size=int(size/2), replace=False)

  return concatenate_datasets([
      entailed_test_dataset.select(entailed_random_indices),
      refuted_test_dataset.select(refuted_random_indices)
  ])

In [ ]:
def preprocess_batch(entries):
    return entries

test_dataset = Dataset.from_pandas(df_test_table_claim)
# test_dataset = get_reduced_random_test_dataset(df_val_table_claim, 100)
# test_dataset = Dataset.from_pandas(df_val_table_claim.head(300))

datasets = DatasetDict({
    "test": test_dataset,
})

tokenized_datasets = datasets.map(preprocess_batch, batched=True, remove_columns="table_claim")